# Topic 4 — Mathematics for ML
### Theory → tiny example → experiment.

You don't need months of pure math first — this notebook gives you just enough **linear algebra,
calculus, statistics, and probability** to understand every algorithm coming up (linear regression,
gradient descent, Naive Bayes, neural nets). Come back to sections as they resurface later.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

## PART A — Linear Algebra

You already used most of this in Topic 1 (NumPy). Here we connect the operations to *why* they matter.

### A1. Vector addition & scalar multiplication

Geometrically: adding vectors chains them head-to-tail; scaling a vector stretches/shrinks it.

In [ ]:
a = np.array([2, 1])
b = np.array([1, 3])

print("a + b =", a + b)        # vector addition
print("3 * a =", 3 * a)        # scalar multiplication

plt.figure(figsize=(4, 4))
plt.quiver(0, 0, a[0], a[1], angles="xy", scale_units="xy", scale=1, color="b", label="a")
plt.quiver(0, 0, b[0], b[1], angles="xy", scale_units="xy", scale=1, color="g", label="b")
plt.quiver(0, 0, (a+b)[0], (a+b)[1], angles="xy", scale_units="xy", scale=1, color="r", label="a+b")
plt.xlim(-1, 5); plt.ylim(-1, 5); plt.legend(); plt.grid(True)
plt.title("Vector addition")
plt.show()

### A2. Dot product, matrix multiplication, transpose, identity, inverse

- **Dot product**: measures alignment between two vectors (used in every weighted sum in ML).
- **Identity matrix**: the "do nothing" matrix, `I @ X = X` (like multiplying by 1).
- **Inverse**: `A @ A_inv = I`. Used in the closed-form solution of linear regression.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print("dot product:", np.dot(a, b))

A = np.array([[2, 0], [1, 2]])
I = np.eye(2)
print("identity:\n", I)
print("A @ I =\n", A @ I)   # unchanged, as expected

A_inv = np.linalg.inv(A)
print("A inverse:\n", A_inv)
print("A @ A_inv =\n", np.round(A @ A_inv, 6))   # should be identity

### A3. Eigenvalues & eigenvectors (basic intuition)

An eigenvector of A is a direction that A only *stretches*, never rotates.
The eigenvalue is how much it stretches by. This is the math behind PCA (Topic 19).

In [ ]:
A = np.array([[2, 0], [0, 3]])
eigvals, eigvecs = np.linalg.eig(A)
print("eigenvalues:", eigvals)
print("eigenvectors:\n", eigvecs)
# Interpretation: A stretches the x-direction by 2x and the y-direction by 3x.
# PCA later finds the eigenvectors of a *covariance* matrix to find directions of max variance.

## PART B — Calculus

ML models "learn" by adjusting parameters to reduce a loss — that adjustment step uses derivatives.

### B1. Derivative — the slope of a function at a point

`f'(x)` tells you: if I nudge `x` slightly, which direction does `f(x)` change, and how fast?

In [ ]:
def f(x):
    return x**2

def f_prime_numeric(x, h=1e-6):
    # numerical derivative: the definition of a derivative, approximated
    return (f(x + h) - f(x - h)) / (2 * h)

for x in [-2, 0, 1, 3]:
    print(f"f'({x}) ≈ {f_prime_numeric(x):.4f}   (analytical: {2*x})")
# f(x) = x^2 has derivative 2x -- confirmed above.

xs = np.linspace(-3, 3, 100)
plt.figure(figsize=(5, 4))
plt.plot(xs, f(xs), label="f(x) = x^2")
plt.axhline(0, color="gray", lw=0.5)
plt.scatter([1], [f(1)], color="red")
plt.title("Slope at x=1 is f'(1)=2 (steep upward)")
plt.legend(); plt.show()

### B2. Partial derivatives & gradient

When a function has multiple inputs (like a model with many weights), the **gradient** is the vector
of partial derivatives — one per input — pointing in the direction of steepest increase.

In [ ]:
def g(x, y):
    return x**2 + y**2

def gradient_numeric(x, y, h=1e-6):
    dgdx = (g(x + h, y) - g(x - h, y)) / (2 * h)
    dgdy = (g(x, y + h) - g(x, y - h)) / (2 * h)
    return np.array([dgdx, dgdy])

print("gradient at (3, 4):", gradient_numeric(3, 4))
# analytically, ∂g/∂x = 2x, ∂g/∂y = 2y -> (6, 8). Matches.

### B3. Gradient descent

The core training loop of nearly every ML/DL model:

```text
repeat:
    compute gradient of loss w.r.t. parameters
    parameters -= learning_rate * gradient
```

Moving *against* the gradient walks downhill toward a minimum of the loss.

In [ ]:
def loss(w):
    return (w - 4) ** 2   # minimum at w = 4

def grad(w):
    return 2 * (w - 4)

w = 0.0            # start far from the minimum
lr = 0.1            # learning rate
history = [w]

for step in range(30):
    w = w - lr * grad(w)
    history.append(w)

print("final w:", w, " (target: 4)")

plt.figure(figsize=(5, 4))
plt.plot(history, marker="o")
plt.xlabel("step"); plt.ylabel("w")
plt.title("Gradient descent converging to w=4")
plt.show()

# --- Try it yourself ---
# Set lr = 1.1 and re-run. Watch it diverge -- this is why learning rate matters.

### B4. Chain rule (why it matters for backprop)

`d/dx f(g(x)) = f'(g(x)) * g'(x)`. Neural nets are long chains of functions —
backpropagation is just the chain rule applied layer by layer, automatically.

In [ ]:
def g(x): return 2 * x + 1
def f(u): return u ** 2

# h(x) = f(g(x)) = (2x+1)^2
def h(x): return f(g(x))

def h_prime_numeric(x, hstep=1e-6):
    return (h(x + hstep) - h(x - hstep)) / (2 * hstep)

x = 3
# chain rule: h'(x) = f'(g(x)) * g'(x) = 2*g(x) * 2 = 4*(2x+1)
analytical = 4 * (2 * x + 1)
print("numeric:", h_prime_numeric(x), " analytical (chain rule):", analytical)

## PART C — Statistics

Describing and summarizing data — used constantly in EDA and evaluating models.

In [ ]:
data = rng.normal(loc=50, scale=10, size=1000)   # e.g. exam scores

print("mean:", np.mean(data))
print("median:", np.median(data))
from scipy import stats
print("mode (binned):", stats.mode(np.round(data), keepdims=True))
print("range:", data.max() - data.min())
print("variance:", np.var(data))          # average squared distance from the mean
print("std dev:", np.std(data))           # sqrt(variance), same units as data
print("25th/50th/75th percentile:", np.percentile(data, [25, 50, 75]))

### C1. Covariance & correlation

**Covariance**: do two variables increase together (positive), move oppositely (negative), or unrelated (~0)?
**Correlation**: covariance rescaled to always be between -1 and +1, so it's comparable across variable pairs.

In [ ]:
x = rng.normal(0, 1, 200)
y = 2 * x + rng.normal(0, 0.5, 200)   # y is roughly linearly related to x

cov_matrix = np.cov(x, y)
corr_matrix = np.corrcoef(x, y)
print("covariance matrix:\n", cov_matrix)
print("correlation matrix:\n", corr_matrix)
print("correlation coefficient:", corr_matrix[0, 1])   # close to +1 here

## PART D — Probability

The mathematical foundation of Naive Bayes and how models express *confidence*, not just a raw prediction.

### D1. Basic probability & independence

In [ ]:
# Simulate flipping a coin 10,000 times
flips = rng.choice(["H", "T"], size=10000)
p_heads = np.mean(flips == "H")
print("P(Heads) ≈", p_heads)   # should be close to 0.5

# Independence: two events don't affect each other's probability
# e.g. two separate coin flips -- P(both heads) should ≈ P(H) * P(H)
flip1 = rng.choice(["H", "T"], size=10000)
flip2 = rng.choice(["H", "T"], size=10000)
p_both = np.mean((flip1 == "H") & (flip2 == "H"))
print("P(both heads) ≈", p_both, " vs P(H)*P(H) =", p_heads**2)

### D2. Conditional probability & Bayes' theorem

`P(A|B)` = probability of A, *given that* B has already happened.

Bayes' theorem:
```text
P(A|B) = P(B|A) * P(A) / P(B)
```
This is literally the algorithm behind Naive Bayes (Topic 11) — e.g.
`P(bullying | word="stupid") = P(word="stupid" | bullying) * P(bullying) / P(word="stupid")`

In [ ]:
# Toy example: a spam-filter-style calculation
# P(spam) = prior probability an email is spam
p_spam = 0.3
# P("free" | spam) = probability the word "free" appears, given the email is spam
p_free_given_spam = 0.6
# P("free" | not spam)
p_free_given_not_spam = 0.05

p_not_spam = 1 - p_spam
# Total probability of seeing "free" at all:
p_free = p_free_given_spam * p_spam + p_free_given_not_spam * p_not_spam

# Bayes' theorem: P(spam | "free")
p_spam_given_free = (p_free_given_spam * p_spam) / p_free
print("P(spam | contains 'free') =", round(p_spam_given_free, 4))
# Seeing "free" massively raises the probability the email is spam, from a 30% prior to this posterior.

### D3. Probability distributions & expected value

In [ ]:
# Normal (Gaussian) distribution
samples = rng.normal(loc=0, scale=1, size=5000)
plt.figure(figsize=(5, 4))
plt.hist(samples, bins=50, density=True)
plt.title("Standard normal distribution")
plt.show()

# Expected value = the probability-weighted average outcome
# Example: a biased die where 6 comes up more often
outcomes = np.array([1, 2, 3, 4, 5, 6])
probs = np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.5])
expected_value = np.sum(outcomes * probs)
print("expected value of the biased die:", expected_value)   # pulled toward 6

## Exercise

Combine the four areas on one tiny dataset.

In [ ]:
scores = rng.normal(65, 12, 300).clip(0, 100)

# --- Try it yourself ---
# 1. Compute mean, median, std, and 90th percentile of `scores`.
# 2. Implement gradient descent to find the minimum of loss(w) = (w - 20)**2, starting from w=100.
# 3. Given P(pass)=0.6, P(studied|pass)=0.8, P(studied|fail)=0.3,
#    use Bayes' theorem to compute P(pass | studied).
# 4. Generate two correlated arrays (like the covariance example) and compute their correlation coefficient.

---
### Next up: **Topic 5 — ML Fundamentals** (supervised/unsupervised learning, bias/variance, overfitting).

Say "next" when you're ready.